<a href="https://colab.research.google.com/github/AlaaTaima/plant-model/blob/alaa/plantmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
color_dir = "/content/drive/MyDrive/plant_project/plantvillage dataset/color"
print(os.path.exists(color_dir), len(os.listdir(color_dir)))   # True 38

Mounted at /content/drive
True 38


In [2]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_a19db74358b723fba726eb9370551b93"

!pip install -q "kaggle>=1.8.0"
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content
!unzip -q /content/plantvillage-dataset.zip "plantvillage dataset/color/*" -d /content

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:10<00:00, 207MB/s]



In [3]:
import os
LOCAL = "/content/plantvillage dataset/color"
classes = sorted(os.listdir(LOCAL))
total = sum(len(os.listdir(os.path.join(LOCAL, c))) for c in classes)
print(len(classes), total)

38 54305


In [4]:
#الدوال و المسار
import os, io, hashlib, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

LOCAL = "/content/plantvillage dataset/color"
classes = sorted(os.listdir(LOCAL))
print("عدد الفئات:", len(classes))

def info(item):
    p, c = item
    row = {"path": p, "label": c, "fname": os.path.basename(p),
           "bytes": os.path.getsize(p), "width": None, "height": None,
           "mode": None, "corrupted": False, "hash": None}
    try:
        with open(p, "rb") as f:
            data = f.read()                      # قراءة واحدة فقط
        row["hash"] = hashlib.md5(data).hexdigest()
        Image.open(io.BytesIO(data)).verify()
        im = Image.open(io.BytesIO(data))
        row["width"], row["height"] = im.size
        row["mode"] = im.mode
    except Exception:
        row["corrupted"] = True
    return row

عدد الفئات: 38


In [5]:
#بناء df وحفظه
items = [(os.path.join(LOCAL, c, f), c)
         for c in classes for f in os.listdir(os.path.join(LOCAL, c))]

with ThreadPoolExecutor(8) as ex:
    rows = list(tqdm(ex.map(info, items), total=len(items)))

df = pd.DataFrame(rows)
df.to_csv("/content/drive/MyDrive/plant_project/df_raw.csv", index=False)
print(df.shape)   # المتوقع (54305, 9)

  0%|          | 0/54305 [00:00<?, ?it/s]

(54305, 9)


In [6]:
#التنظيف والحفظ
ok = df[~df["corrupted"]]
dup = ok.duplicated(subset="hash", keep="first")
n_labels = ok.groupby("hash")["label"].nunique()
conflict = n_labels[n_labels > 1].index

bad = set(df.loc[df["corrupted"], "path"])
bad |= set(ok.loc[dup, "path"])
bad |= set(ok.loc[ok["hash"].isin(conflict), "path"])

clean_df = df[~df["path"].isin(bad)].drop(columns="corrupted").reset_index(drop=True)
print(len(df), "|", len(bad), "|", len(clean_df))

clean_df.to_csv("/content/drive/MyDrive/plant_project/clean_df.csv", index=False)

54305 | 21 | 54284


In [8]:
#حساب pHash
!pip install -q imagehash

import numpy as np, pandas as pd, imagehash
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

clean_df = pd.read_csv("/content/drive/MyDrive/plant_project/clean_df.csv")

def to_u64(h):
    return np.packbits(h.hash.flatten()).view(">u8")[0].astype(np.uint64)

def variants(im):
    out = []
    for flip in (False, True):
        b = im.transpose(Image.FLIP_LEFT_RIGHT) if flip else im
        for r in (0, 90, 180, 270):
            out.append(b.rotate(r, expand=True) if r else b)
    return out

def phashes(p):
    im = Image.open(p).convert("RGB")
    return [to_u64(imagehash.phash(v)) for v in variants(im)]

with ThreadPoolExecutor(8) as ex:
    H = np.array(list(tqdm(ex.map(phashes, clean_df["path"]), total=len(clean_df))),
                 dtype=np.uint64)          # الشكل: (N, 8)

np.save("/content/drive/MyDrive/plant_project/phash.npy", H)   # احفظها كي لا تعيد الحساب
print(H.shape)

  0%|          | 0/54284 [00:00<?, ?it/s]

(54284, 8)


In [13]:
#تجميع الصور المتشابهة في مجموعات
#المقارنة تتم داخل كل فئة فقط (أسرع بكثير، ونفس الورقة تكون دائماً في نفس الفئة).

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def popcount(x):
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    x = np.ascontiguousarray(x)
    return np.unpackbits(x.view(np.uint8).reshape(x.shape + (8,)), axis=-1).sum(-1)

THRESH = 2       # أقصى فرق مسموح (من 64 بت). ابدأ بـ 6
rows, cols = [], []

for label, idx in tqdm(clean_df.groupby("label").indices.items()):
    h = H[idx]                    # (n, 8)
    h0 = h[:, 0]
    for s in range(0, len(idx), 200):
        x = h0[s:s+200, None, None] ^ h[None, :, :]       # (c, n, 8)
        d = popcount(x).min(axis=2)                       # أقل فرق بين كل الاتجاهات
        a, b = np.nonzero(d <= THRESH)
        rows.append(idx[a + s]); cols.append(idx[b])

rows, cols = np.concatenate(rows), np.concatenate(cols)
N = len(clean_df)
graph = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))
n_groups, groups = connected_components(graph, directed=False)
clean_df["group"] = groups

sizes = clean_df["group"].value_counts()
print("عدد المجموعات:", n_groups)
print("مجموعات فيها أكثر من صورة:", (sizes > 1).sum())
print("أكبر مجموعة:", sizes.max())

  0%|          | 0/38 [00:00<?, ?it/s]

عدد المجموعات: 54019
مجموعات فيها أكثر من صورة: 159
أكبر مجموعة: 27


In [15]:
from sklearn.model_selection import StratifiedGroupKFold

# 1/7 ≈ 14.3% للاختبار
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
tv_idx, test_idx = next(sgkf.split(clean_df, clean_df["label"], clean_df["group"]))
tv, test_df = clean_df.iloc[tv_idx], clean_df.iloc[test_idx]

# 1/6 من الباقي ≈ 14.3% للتحقق
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=42)
tr_i, va_i = next(sgkf2.split(tv, tv["label"], tv["group"]))
train_df, val_df = tv.iloc[tr_i], tv.iloc[va_i]

# فحوصات
g = lambda d: set(d["group"])
print("تداخل train/val :", len(g(train_df) & g(val_df)))
print("تداخل train/test:", len(g(train_df) & g(test_df)))
print("تداخل val/test  :", len(g(val_df) & g(test_df)))
print("الأحجام:", len(train_df), len(val_df), len(test_df))
print("أقل عدد صور لفئة في test:", test_df["label"].value_counts().min())

d = "/content/drive/MyDrive/plant_project/"
train_df.to_csv(d + "train.csv", index=False)
val_df.to_csv(d + "val.csv", index=False)
test_df.to_csv(d + "test.csv", index=False)

تداخل train/val : 0
تداخل train/test: 0
تداخل val/test  : 0
الأحجام: 38777 7760 7747
أقل عدد صور لفئة في test: 22


In [3]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
import pandas as pd
d = "/content/drive/MyDrive/plant_project/"
train_df = pd.read_csv(d + "train.csv")
val_df = pd.read_csv(d + "val.csv")
test_df = pd.read_csv(d + "test.csv")
print(len(train_df), len(val_df), len(test_df))   # 38777 7760 7747

38777 7760 7747


In [15]:
#الإعدادات وترميز الفئات
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG = 128
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

classes = sorted(train_df["label"].unique())
class2idx = {c: i for i, c in enumerate(classes)}
for d in (train_df, val_df, test_df):
    d["y"] = d["label"].map(class2idx)
NUM_CLASSES = len(classes)
print(NUM_CLASSES)   # 38

38


In [16]:
#pipeline الصور (tf.data)
def load(path, y):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG, IMG))
    return img, y          # التطبيع يتم داخل المودل

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["y"].values))
    if training:
        ds = ds.shuffle(len(df), seed=42)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds   = make_ds(val_df, False)
test_ds  = make_ds(test_df, False)

In [17]:
#بناء المودل Transfer Learning

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augment")

inputs = keras.Input((IMG, IMG, 3))
x = augment(inputs)
x = layers.Rescaling(1./255)(x)

for filters in (32, 64, 128, 256):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

cnn = keras.Model(inputs, outputs)
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augment (Sequential)            │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 256)    │       295,16

 Total params: 1,251,654 (4.77 MB)

 Trainable params: 1,249,734 (4.77 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [ ]:
#التدريب ()


cb2 = [
    keras.callbacks.ModelCheckpoint("/content/drive/MyDrive/plant_project/cnn_best.keras",
                                    monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
]
h_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cb2)

Epoch 1/30
606/606 ━━━━━━━━━━━━━━━━━━━━ 152s 229ms/step - accuracy: 0.6228 - loss: 1.3162 - val_accuracy: 0.1515 - val_loss: 7.3224 - learning_rate: 0.0010
Epoch 2/30
606/606 ━━━━━━━━━━━━━━━━━━━━ 137s 226ms/step - accuracy: 0.8034 - loss: 0.6320 - val_accuracy: 0.7312 - val_loss: 0.8963 - learning_rate: 0.0010
Epoch 3/30
606/606 ━━━━━━━━━━━━━━━━━━━━ 133s 219ms/step - accuracy: 0.8715 - loss: 0.4068 - val_accuracy: 0.7036 - val_loss: 1.0377 - learning_rate: 0.0010
Epoch 4/30
569/606 ━━━━━━━━━━━━━━━━━━━━ 7s 207ms/step - accuracy: 0.8960 - loss: 0.3238

In [ ]:
#التقييم النهائي على test (مرة واحدة فقط)

from sklearn.metrics import classification_report

cnn = keras.models.load_model("/content/drive/MyDrive/plant_project/cnn_best.keras")

loss, acc = cnn.evaluate(test_ds)
print("Test accuracy:", acc)

y_true = test_df["y"].values
y_pred = np.argmax(cnn.predict(test_ds), axis=1)
print(classification_report(y_true, y_pred, target_names=classes, digits=3))